# Attention Weights as a Heatmap

Attention learns **which positions to focus on** when producing each output.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (10, 5)
torch.manual_seed(42); np.random.seed(42)


In [ ]:
class DummyDataGenerator:
    def __init__(self, seq_len=8, d=16):
        self.q = torch.randn(1, seq_len, d)
        self.k = torch.randn(1, seq_len, d)
        self.v = torch.randn(1, seq_len, d)
    def qkv(self): return self.q, self.k, self.v

class AttentionModel(nn.Module):
    def __init__(self, d=16):
        super().__init__()
        self.d = d
    def forward(self, q, k, v):
        scores = (q @ k.transpose(-2, -1)) / (self.d ** 0.5)
        weights = torch.softmax(scores, dim=-1)
        out = weights @ v
        return out, weights

q, k, v = DummyDataGenerator().qkv()
model = AttentionModel(d=16)
out, weights = model(q, k, v)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
im = axes[0].imshow(weights[0].detach().numpy(), cmap='hot', aspect='auto')
axes[0].set_xlabel('Key position'); axes[0].set_ylabel('Query position')
axes[0].set_title('Attention weights (softmax of QK^T/√d)')
plt.colorbar(im, ax=axes[0])
axes[1].bar(range(weights.shape[-1]), weights[0, -1].detach().numpy(), color='steelblue')
axes[1].set_title('Last query attends to these keys'); axes[1].set_xlabel('key index')
plt.tight_layout(); plt.show()